# EfficientMatch -- Expérience FlexMatch (baseline)

Ce notebook implémente **FlexMatch** (Zhang et al., 2021) selon le protocole réduit décrit dans le papier (budget $2^{17}$ itérations au lieu de $2^{20}$, schedule cosine recalé).

**Structure volontaire** :
- Toutes les sections (data, modèle, EMA, évaluation, boucle principale) sont génériques et réutilisables telles quelles pour FixMatch.
- **Seule la fonction `train_step_flexmatch`** contient la logique spécifique à FlexMatch (seuillage adaptatif par classe, Curriculum Pseudo Labeling). Pour basculer vers FixMatch, il suffit d'écrire une fonction `train_step_fixmatch` avec un seuil fixe et de changer l'appel dans la boucle principale (section 8) -- rien d'autre à toucher.
- Pas de superclasse ni d'abstraction : chaque ligne de l'algorithme (batch labellisé, batch non labellisé faible/fort, pseudo-étiquetage, masque, pertes, backward) est visible et explicite dans `train_step_flexmatch`.
- Les optimisations de vitesse (AMP, subset de debug, cudnn benchmark, etc.) sont **toutes débrayables** via le dictionnaire `CONFIG` en section 1, contrairement au protocole USB qui les fixe implicitement.

## 1. Imports

In [ ]:
import os
import time
import json
import math
import random
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms_v1
import torchvision.transforms.v2 as transforms_v2

print(f"Torch version: {torch.__version__}, CUDA disponible: {torch.cuda.is_available()}")

## 2. Configuration

Tout est centralisé ici. Les clés sous `# --- OPTIMISATIONS DE VITESSE ---` sont **débrayables indépendamment** : chacune peut être mise à `False`/valeur par défaut pour retrouver un comportement proche du protocole USB standard (plus lent, mais plus proche de la référence), utile pour un test de non-régression.

In [ ]:
CONFIG = {
    # --- Dataset ---
    "dataset": "cifar10",          # "cifar10" | "cifar100" | "pathmnist"
    "data_root": "./data",
    "n_labels": 40,                 # budget de labels total (régime faible labellisation)
    "num_classes": 10,

    # --- Hyperparamètres standards SSL (cf. Tableau hyperparamètres du papier) ---
    "B": 64,                        # batch size labellisé
    "mu": 7,                        # ratio non labellisé / labellisé
    "lr": 0.03,
    "momentum": 0.9,
    "nesterov": True,
    "weight_decay": 5e-4,
    "tau": 0.95,                    # seuil de base (utilisé pour normaliser le seuil adaptatif FlexMatch)
    "lambda_u": 1.0,
    "ema_decay": 0.999,

    # --- Budget d'entraînement (protocole réduit, cf. papier) ---
    "K": 2 ** 17,                   # nombre total d'itérations (au lieu de 2**20 standard)
    "iters_per_epoch": 1024,        # convention standard FixMatch/FlexMatch
    "eval_every": 512,              # évaluation toutes les N itérations (pour tracer accuracy vs itérations)
    "seed": 0,

    # --- Early stopping (plateau de l'accuracy EMA, cf. detect_plateau) ---
    "early_stopping": True,
    "es_window": 5,
    "es_slope_threshold": 1e-4,

    # --- OPTIMISATIONS DE VITESSE (débrayables) ---
    "use_amp": True,                # mixed precision (torch.autocast + GradScaler)
    "cudnn_benchmark": True,        # cudnn.benchmark = True (gain si tailles d'input fixes)
    "channels_last": True,          # format mémoire channels_last (gain sur convs, GPU récents)
    "use_transforms_v2": True,      # True = torchvision.transforms.v2 (batch vectorisé) / False = v1 classique (par image)
    "num_workers": 4,               # workers DataLoader (mettre 0 pour debug)
    "persistent_workers": True,
    "pin_memory": True,
    "debug_subset_size": None,      # ex: 2000 pour tester le pipeline sur un mini sous-ensemble ; None = désactivé
    "compile_model": False,         # torch.compile() -- gain potentiel mais coût de compilation initial

    # --- Divers ---
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "log_path": "./logs_flexmatch.json",
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

if CONFIG["cudnn_benchmark"]:
    torch.backends.cudnn.benchmark = True  # OPTIMISATION : accélère si les tailles de tenseurs sont stables

device = torch.device(CONFIG["device"])
print("Config chargée. Device:", device)

## 3. Traitement des données

On définit :
1. Les augmentations **faible** ($\alpha$) et **forte** ($\mathcal{A}$, RandAugment).
2. Le split labellisé / non labellisé (répartition équilibrée par classe pour les labels).
3. Un `Dataset` qui retourne **trois vues** pour les données non labellisées (une pour l'éval interne, une faible, une forte) -- géré ici en retournant l'image brute et en appliquant les transformations à la volée dans la boucle pour plus de clarté.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2471, 0.2435, 0.2616)

USE_TRANSFORMS_V2 = CONFIG["use_transforms_v2"]
T = transforms_v2 if USE_TRANSFORMS_V2 else transforms_v1

if USE_TRANSFORMS_V2:
    # Augmentation FAIBLE (alpha) : flip horizontal + translation aléatoire, standard FixMatch/FlexMatch
    # v2 : un seul appel vectorisé sur tout le batch (CPU ou GPU), plus de boucle Python par image
    weak_transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    # Augmentation FORTE (A) : RandAugment, comme dans FixMatch/FlexMatch
    strong_transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.RandAugment(num_ops=2, magnitude=10),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
        T.RandomErasing(p=0.5),  # Cutout-like, standard dans les implémentations FixMatch
    ])

    eval_transform = T.Compose([
        T.PILToTensor(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
else:
    # v1 (classique) : transform appliqué image par image (boucle Python) dans la boucle d'entraînement
    weak_transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    strong_transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.RandAugment(num_ops=2, magnitude=10),
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
        T.RandomErasing(p=0.5),
    ])

    eval_transform = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])


def apply_batch(transform, raw_batch):
    """Bascule transparente v1 (liste de PIL, boucle Python) / v2 (batch tenseur vectorisé)."""
    if USE_TRANSFORMS_V2:
        return transform(raw_batch.to(device, non_blocking=True))
    return torch.stack([transform(img) for img in raw_batch]).to(device, non_blocking=True)


def ssl_collate(batch):
    """Collate custom : le DataLoader ne sait pas empiler nativement une liste d'images PIL (mode v1)."""
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs) if USE_TRANSFORMS_V2 else list(imgs)
    return imgs, torch.tensor(labels)


class SSLDataset(Dataset):
    """Dataset générique retournant une image (PIL en mode v1, tenseur uint8 CHW en mode v2) + son label
    (-1 si non labellisé). Les transformations faible/forte sont appliquées à l'extérieur (dans le
    training step), pour bien montrer explicitement dans la boucle que deux vues distinctes sont générées.
    """
    def __init__(self, base_dataset, indices):
        self.base_dataset = base_dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.base_dataset[self.indices[idx]]
        if USE_TRANSFORMS_V2:
            img = transforms_v2.functional.pil_to_tensor(img)  # uint8 CHW -> collate direct en batch tenseur
        return img, label


def make_ssl_split(base_dataset, n_labels, num_classes, seed=0):
    """Split équilibré par classe : n_labels au total, répartis également entre classes."""
    rng = np.random.RandomState(seed)
    targets = np.array(base_dataset.targets)
    n_per_class = n_labels // num_classes
    labeled_idx = []
    for c in range(num_classes):
        idx_c = np.where(targets == c)[0]
        rng.shuffle(idx_c)
        labeled_idx.extend(idx_c[:n_per_class].tolist())
    labeled_idx = np.array(labeled_idx)
    unlabeled_idx = np.arange(len(base_dataset))  # tout le dataset sert de pool non labellisé (standard SSL)
    return labeled_idx, unlabeled_idx


def load_datasets(cfg):
    if cfg["dataset"] == "cifar10":
        train_base = torchvision.datasets.CIFAR10(cfg["data_root"], train=True, download=True)
        test_base = torchvision.datasets.CIFAR10(cfg["data_root"], train=False, download=True, transform=eval_transform)
    elif cfg["dataset"] == "cifar100":
        train_base = torchvision.datasets.CIFAR100(cfg["data_root"], train=True, download=True)
        test_base = torchvision.datasets.CIFAR100(cfg["data_root"], train=False, download=True, transform=eval_transform)
    else:
        raise NotImplementedError(
            f"Dataset {cfg['dataset']} non branché ici -- ajouter le chargement MedMNIST (PathMNIST) via medmnist.PathMNIST"
        )

    labeled_idx, unlabeled_idx = make_ssl_split(train_base, cfg["n_labels"], cfg["num_classes"], seed=cfg["seed"])

    # OPTIMISATION (débrayable) : sous-ensemble de debug pour itérer vite sur le pipeline
    if cfg["debug_subset_size"] is not None:
        unlabeled_idx = unlabeled_idx[: cfg["debug_subset_size"]]
        test_base = Subset(test_base, list(range(min(len(test_base), cfg["debug_subset_size"]))))

    labeled_set = SSLDataset(train_base, labeled_idx)
    unlabeled_set = SSLDataset(train_base, unlabeled_idx)

    return labeled_set, unlabeled_set, test_base


def infinite_loader(dataset, batch_size, cfg, shuffle=True):
    """Itérateur infini sur un DataLoader (pratique car labellisé/non-labellisé n'ont pas la même taille)."""
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg["num_workers"],
        pin_memory=cfg["pin_memory"],
        persistent_workers=cfg["persistent_workers"] and cfg["num_workers"] > 0,
        drop_last=True,
        collate_fn=ssl_collate,
    )
    while True:
        for batch in loader:
            yield batch

## 4. Modèle : WideResNet-28-2

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, drop_rate=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu1 = nn.LeakyReLU(0.1, inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.relu2 = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, 3, stride=1, padding=1, bias=False)
        self.drop_rate = drop_rate
        self.equal_io = in_planes == out_planes and stride == 1
        self.shortcut = None if self.equal_io else nn.Conv2d(in_planes, out_planes, 1, stride=stride, bias=False)

    def forward(self, x):
        out = self.relu1(self.bn1(x))
        shortcut = x if self.equal_io else self.shortcut(out)
        out = self.conv1(out)
        out = self.relu2(self.bn2(out))
        if self.drop_rate > 0:
            out = F.dropout(out, p=self.drop_rate, training=self.training)
        out = self.conv2(out)
        return out + shortcut


class WideResNet(nn.Module):
    """WideResNet-28-widen_factor, implémentation standard utilisée dans FixMatch/FlexMatch."""
    def __init__(self, num_classes=10, depth=28, widen_factor=2, drop_rate=0.0):
        super().__init__()
        n_channels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6

        self.conv1 = nn.Conv2d(3, n_channels[0], 3, stride=1, padding=1, bias=False)
        self.block1 = self._make_block(n_channels[0], n_channels[1], n, stride=1, drop_rate=drop_rate)
        self.block2 = self._make_block(n_channels[1], n_channels[2], n, stride=2, drop_rate=drop_rate)
        self.block3 = self._make_block(n_channels[2], n_channels[3], n, stride=2, drop_rate=drop_rate)
        self.bn1 = nn.BatchNorm2d(n_channels[3])
        self.relu = nn.LeakyReLU(0.1, inplace=True)
        self.fc = nn.Linear(n_channels[3], num_classes)
        self.n_channels = n_channels[3]

    def _make_block(self, in_planes, out_planes, num_layers, stride, drop_rate):
        layers = [BasicBlock(in_planes, out_planes, stride, drop_rate)]
        for _ in range(1, num_layers):
            layers.append(BasicBlock(out_planes, out_planes, 1, drop_rate))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)


def build_model(cfg):
    model = WideResNet(num_classes=cfg["num_classes"], depth=28, widen_factor=2)
    model = model.to(device)
    if cfg["channels_last"]:
        model = model.to(memory_format=torch.channels_last)  # OPTIMISATION : layout mémoire plus rapide sur GPU récents
    if cfg["compile_model"]:
        model = torch.compile(model)  # OPTIMISATION : compilation JIT, coût initial mais accélération ensuite
    return model

## 5. EMA (modèle d'évaluation) et estimation de FLOPs

In [ ]:
class EMA:
    """Moyenne mobile exponentielle des poids, utilisée pour l'évaluation (standard FixMatch/FlexMatch)."""
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def estimate_flops_per_iter(cfg):
    """Estimation simplifiée des FLOPs par itération, comme proxy indépendant du hardware
    (cf. Fast FixMatch). Compte les forward passes : 1x batch labellisé + mu*2 batch non labellisé
    (vue faible + vue forte) + mu passes supplémentaires pour le canal de Mixup filtré si activé.
    Ici on rapporte un COMPTE DE FORWARD PASSES plutôt qu'un vrai calcul de FLOPs (nécessiterait
    thop/fvcore sur le modèle exact) -- à affiner si une mesure plus précise est nécessaire.
    """
    n_forward_labeled = cfg["B"]
    n_forward_unlabeled = cfg["mu"] * cfg["B"] * 2  # vue faible + vue forte
    return n_forward_labeled + n_forward_unlabeled


@torch.no_grad()
def evaluate(model, test_loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    model.train()
    return correct / total

## 6. Boucle d'entraînement -- FlexMatch

Chaque ligne de l'algorithme est explicite ci-dessous : batch labellisé, batch non labellisé (vues faible/forte), pseudo-étiquetage, **seuillage adaptatif par classe (Curriculum Pseudo Labeling)**, pertes, backward.

**Mécanisme spécifique à FlexMatch** : au lieu d'un seuil fixe $\tau$ (FixMatch), on maintient un compteur $\sigma_t(c)$ du nombre d'exemples non labellisés dont la pseudo-étiquette de classe $c$ dépasse $\tau$ à l'itération courante, puis on normalise par la classe la plus apprise pour obtenir $\beta_t(c) = \sigma_t(c) / \max_c \sigma_t(c)$, et le seuil effectif par classe est $\tau_t(c) = \dfrac{\beta_t(c)}{2 - \beta_t(c)} \cdot \tau$.

In [ ]:
def train_step_flexmatch(model, ema, optimizer, scaler, class_counts, cfg,
                          labeled_iter, unlabeled_iter):
    """Une itération d'entraînement FlexMatch. Pour basculer vers FixMatch :
    remplacer le bloc \"SEUIL ADAPTATIF\" ci-dessous par un seuil fixe `cfg['tau']`
    et supprimer la mise à jour de `class_counts` -- rien d'autre à changer.
    """
    # --- Étape 1 : batch labellisé ---
    imgs_x_raw, labels_x = next(labeled_iter)
    imgs_x = apply_batch(weak_transform, imgs_x_raw)
    labels_x = labels_x.to(device, non_blocking=True)

    # --- Étape 2 : batch non labellisé, deux vues (faible / forte) ---
    imgs_u_raw, _ = next(unlabeled_iter)  # le vrai label n'est jamais utilisé pendant l'entraînement
    imgs_u_w = apply_batch(weak_transform, imgs_u_raw)
    imgs_u_s = apply_batch(strong_transform, imgs_u_raw)

    if cfg["channels_last"]:
        imgs_x = imgs_x.to(memory_format=torch.channels_last)
        imgs_u_w = imgs_u_w.to(memory_format=torch.channels_last)
        imgs_u_s = imgs_u_s.to(memory_format=torch.channels_last)

    optimizer.zero_grad(set_to_none=True)

    # OPTIMISATION (débrayable) : mixed precision autocast
    with torch.autocast(device_type=cfg["device"], enabled=cfg["use_amp"]):

        # --- Étape 3 : perte supervisée ---
        logits_x = model(imgs_x)
        loss_s = F.cross_entropy(logits_x, labels_x)

        # --- Étape 4 : pseudo-étiquetage sur la vue faible (sans gradient) ---
        with torch.no_grad():
            logits_u_w = model(imgs_u_w)
            probs_u_w = F.softmax(logits_u_w, dim=-1)
            max_probs, pseudo_labels = probs_u_w.max(dim=-1)

        # --- Étape 5 : SEUIL ADAPTATIF (spécifique FlexMatch -- Curriculum Pseudo Labeling) ---
        # 5a. compter, pour chaque classe, le nombre d'exemples dont la confiance dépasse tau
        with torch.no_grad():
            confident_mask_raw = max_probs.ge(cfg["tau"])
            for c in range(cfg["num_classes"]):
                class_counts[c] += ((pseudo_labels == c) & confident_mask_raw).sum().item()

            # 5b. normaliser par la classe la plus représentée (beta_t)
            max_count = max(max(class_counts.values()), 1)  # évite division par zéro au tout début
            beta_t = {c: class_counts[c] / max_count for c in range(cfg["num_classes"])}

            # 5c. seuil effectif par classe : tau_t(c) = beta_t(c) / (2 - beta_t(c)) * tau
            tau_per_class = torch.tensor(
                [beta_t[c] / (2 - beta_t[c]) * cfg["tau"] for c in range(cfg["num_classes"])],
                device=device,
            )

            # 5d. masque final : la pseudo-étiquette est retenue si sa confiance dépasse
            #     le seuil ADAPTATIF de SA PROPRE classe (et non le tau fixe global)
            threshold_per_sample = tau_per_class[pseudo_labels]
            mask = max_probs.ge(threshold_per_sample).float()

        # --- Étape 6 : perte de cohérence faible/forte, filtrée par le masque ---
        logits_u_s = model(imgs_u_s)
        loss_u_per_sample = F.cross_entropy(logits_u_s, pseudo_labels, reduction="none")
        loss_u = (loss_u_per_sample * mask).mean()

        # --- Étape 7 : perte totale ---
        loss = loss_s + cfg["lambda_u"] * loss_u

    # --- Étape 8 : backward + optimisation (avec ou sans GradScaler selon AMP) ---
    if cfg["use_amp"]:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()

    # --- Étape 9 : mise à jour EMA ---
    ema.update(model)

    return {
        "loss": loss.item(),
        "loss_s": loss_s.item(),
        "loss_u": loss_u.item(),
        "mask_rate": mask.mean().item(),  # fraction de pseudo-labels retenus -- utile à monitorer
    }

## 7. Assemblage : modèle, optimiseur, scheduler

In [ ]:
def cosine_schedule(optimizer, k, K):
    """eta = eta_0 * cos(7*pi*k / (16*K)), K recalé sur le budget réduit (cf. papier, section protocole)."""
    base_lr = optimizer.defaults["lr"]
    new_lr = base_lr * math.cos(7 * math.pi * k / (16 * K))
    for group in optimizer.param_groups:
        group["lr"] = max(new_lr, 0.0)


labeled_set, unlabeled_set, test_set = load_datasets(CONFIG)

labeled_iter = infinite_loader(labeled_set, CONFIG["B"], CONFIG, shuffle=True)
unlabeled_iter = infinite_loader(unlabeled_set, CONFIG["mu"] * CONFIG["B"], CONFIG, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=CONFIG["num_workers"])

model = build_model(CONFIG)
ema = EMA(model, CONFIG["ema_decay"])

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=CONFIG["lr"],
    momentum=CONFIG["momentum"],
    nesterov=CONFIG["nesterov"],
    weight_decay=CONFIG["weight_decay"],
)

scaler = torch.amp.GradScaler(enabled=CONFIG["use_amp"])  # OPTIMISATION : no-op si use_amp=False

class_counts = {c: 0 for c in range(CONFIG["num_classes"])}  # état FlexMatch, persiste entre itérations

print(f"FLOPs (proxy forward passes) par itération : {estimate_flops_per_iter(CONFIG)}")
print(f"Budget total : {CONFIG['K']} itérations")

In [ ]:
def detect_plateau(acc_history, window=5, slope_threshold=1e-4):
    """Détecte un plateau via la pente d'une régression linéaire locale sur l'accuracy EMA.
    Préféré à un compteur de patience car la pente s'adapte à l'échelle locale du bruit,
    plutôt que de dépendre d'un seuil absolu sur l'accuracy (cf. justification papier :
    argument tiré de la décroissance du schedule cosine, qui porte sur un TAUX de variation).
    """
    if len(acc_history) < window:
        return False, None
    recent = np.array(acc_history[-window:])
    x = np.arange(window)
    slope = np.polyfit(x, recent, 1)[0]
    return abs(slope) < slope_threshold, slope

## 8. Boucle principale + logging

Enregistre à chaque évaluation : itération, temps écoulé, FLOPs cumulés, accuracy (modèle EMA), taux de masque -- directement exploitable pour les figures accuracy-vs-itérations / accuracy-vs-FLOPs du papier.

In [ ]:
logs = []
acc_history = []
flops_per_iter = estimate_flops_per_iter(CONFIG)
eval_model = build_model(CONFIG)  # copie séparée pour l'évaluation avec les poids EMA, sans perturber `model`

start_time = time.time()
model.train()

for k in range(1, CONFIG["K"] + 1):
    cosine_schedule(optimizer, k, CONFIG["K"])

    # >>> Appel de la fonction spécifique à l'algorithme -- SEULE LIGNE À CHANGER POUR PASSER À FIXMATCH <<<
    step_metrics = train_step_flexmatch(model, ema, optimizer, scaler, class_counts, CONFIG,
                                         labeled_iter, unlabeled_iter)

    if k % CONFIG["eval_every"] == 0 or k == CONFIG["K"]:
        ema.copy_to(eval_model)
        acc = evaluate(eval_model, test_loader)
        elapsed = time.time() - start_time
        cumulative_flops = flops_per_iter * k

        log_entry = {
            "iteration": k,
            "elapsed_seconds": elapsed,
            "cumulative_flops_proxy": cumulative_flops,
            "eval_accuracy": acc,
            **step_metrics,
        }
        logs.append(log_entry)
        print(f"[iter {k:>7}/{CONFIG['K']}] acc={acc:.4f} loss={step_metrics['loss']:.4f} "
              f"mask_rate={step_metrics['mask_rate']:.3f} elapsed={elapsed/60:.1f}min")

        # sauvegarde incrémentale -- utile pour reprendre/monitorer sur un run long
        with open(CONFIG["log_path"], "w") as f:
            json.dump({"config": CONFIG, "logs": logs}, f, indent=2)

        acc_history.append(acc)
        if CONFIG["early_stopping"]:
            is_plateau, slope = detect_plateau(acc_history, CONFIG["es_window"], CONFIG["es_slope_threshold"])
            if is_plateau:
                print(f"Plateau détecté (pente={slope:.2e} < seuil={CONFIG['es_slope_threshold']:.2e}) "
                      f"-- arrêt anticipé à l'itération {k}.")
                break

print("Entraînement terminé.")

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
import torch
import torch.nn.functional as F

def estimate_flops_per_iter(model, cfg):
    """Mesure le nombre réel de FLOPs (forward + backward) pour une itération d'entraînement,
    via FlopCounterMode englobant un vrai passage forward+backward. Exécuté UNE SEULE FOIS
    avant l'entraînement sur des tenseurs factices -- ne pas appeler pendant la boucle.
    """
    device = next(model.parameters()).device
    model.train()  # même mode que pendant l'entraînement réel (BN/dropout actifs)

    dummy_x = torch.randn(cfg["B"], 3, 32, 32, device=device, requires_grad=False)
    dummy_u_w = torch.randn(cfg["mu"] * cfg["B"], 3, 32, 32, device=device, requires_grad=False)
    dummy_u_s = torch.randn(cfg["mu"] * cfg["B"], 3, 32, 32, device=device, requires_grad=False)

    dummy_labels_x = torch.randint(0, cfg["num_classes"], (cfg["B"],), device=device)
    dummy_labels_u = torch.randint(0, cfg["num_classes"], (cfg["mu"] * cfg["B"],), device=device)

    model.zero_grad(set_to_none=True)

    with FlopCounterMode(display=False) as flop_counter:
        # forward des 3 passages, comme dans train_step_flexmatch
        logits_x = model(dummy_x)
        logits_u_w = model(dummy_u_w)
        logits_u_s = model(dummy_u_s)

        # perte factice de même forme que la vraie perte totale (nécessaire pour un backward réaliste)
        loss = F.cross_entropy(logits_x, dummy_labels_x) + F.cross_entropy(logits_u_s, dummy_labels_u)
        loss.backward()

    model.zero_grad(set_to_none=True)  # nettoyage, les gradients factices ne doivent pas polluer l'entraînement
    return flop_counter.get_total_flops()

## 9. Notes pour la bascule vers FixMatch

Pour créer la version FixMatch sans dupliquer tout le notebook :
1. Copier ce fichier, renommer en `fixmatch_experiment.ipynb`.
2. Dans la fonction (renommée `train_step_fixmatch`), remplacer intégralement le bloc **Étape 5 (SEUIL ADAPTATIF)** par une seule ligne :
   ```python
   mask = max_probs.ge(cfg["tau"]).float()
   ```
3. Supprimer l'argument `class_counts` (devenu inutile) de la signature et de l'appel en section 8.
4. Tout le reste (data, modèle, EMA, boucle principale, logging) reste strictement identique -- ce qui garantit que toute différence de résultat entre les deux notebooks provient uniquement du mécanisme de seuillage, et non d'une différence incidente de pipeline.